# Census MTIS 2018–2022 原始来源格式与传输修复

冻结完整Stage1 f696+b472父方案；本卡从NAS已备份59份原件与一个旧样本出发，只针对16份Not Adjusted大小写变化修分隔符，对3份部分下载原址各重取一次。原先41个合格月份必须逐字段不变。零SOXX结果、拟合、账户。

In [1]:
import datetime,email.utils,hashlib,io,json,os,pathlib,re,shlex,subprocess,sys,tarfile,tempfile
from pypdf import PdfReader
import pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_2018_2022_source_repair_20261023';snap=root/'data/snapshots/stage1_census_mtis_2018_2022_source_repair_20261023';snap.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text());s=plan['sources'];sha=lambda x:hashlib.sha256(pathlib.Path(x).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists();old_card=root/s['prior_card'];old_summary=json.loads((old_card/'summary.json').read_text());old_result=json.loads((old_card/'result.json').read_text());old_snap=root/'data/snapshots/stage1_census_mtis_2018_2022_original_source_20261023';assert sha(old_card/'summary.json')==s['prior_summary_sha256'] and sha(old_card/'result.json')==s['prior_result_sha256'] and sha(old_snap/'response_receipts.json')==s['prior_receipts_sha256'] and old_summary['source_NAS_sha256']==s['prior_source_NAS_sha256'] and old_summary['source_NAS_cold_restore_verified'];assert sha(root/'data/snapshots/stage1_census_mtis_index_transport_repair_20261023/official_index.response')==s['saved_official_index_sha256'];assert sorted(old_result['unknown_months'])==sorted(s['transport_months_exact']+s['case_variant_months_exact'])
receipts=json.loads((old_snap/'response_receipts.json').read_text());by_month={r['month']:r for r in receipts};assert len(by_month)==60
state={'status':'three_exact_GETs_started','plan_sha256':sha(card/'plan.json'),'external_GETs_started':0,'saved_PDF_reads':0,'prior_Recorder_object_loads':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');new={}
for month in s['transport_months_exact']:
 u=by_month[month]['url'];assert u==f'https://www2.census.gov/mtis/historical/mtis{int(month[:4])%100:02d}{int(month[5:]):02d}.pdf';state['external_GETs_started']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');body=snap/(month+'.response');headers=snap/(month+'.headers');start=datetime.datetime.now(datetime.timezone.utc).isoformat();proc=subprocess.run(['curl','--http1.1','--location','--silent','--show-error','--connect-timeout','5','--max-time','35','--user-agent','soxl-qlib-source-research','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',u],capture_output=True,text=True);parts=proc.stdout.strip().split('|',1);new[month]={'month':month,'url':u,'started_utc':start,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'curl_exit':proc.returncode,'http_status':int(parts[0]) if parts and parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'error':proc.stderr[:250] or None,'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'local_path':str(body.relative_to(root)) if body.exists() else None,'headers_path':str(headers.relative_to(root)) if headers.exists() else None,'headers_sha256':sha(headers) if headers.exists() else None,'within_size_cap':body.exists() and body.stat().st_size<=2000000}
(snap/'repair_receipts.json').write_text(json.dumps(new,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';provider=str(root/'data/qlib/soxl');prior_uri=str(old_card/'qlib_recorder');qlib.init(provider_uri=provider,region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':prior_uri,'default_exp_name':old_card.name}});prior=R.get_recorder(experiment_id=s['prior_experiment_id'],recorder_id=s['prior_recorder_id']);assert prior.status=='FINISHED';old_table=prior.load_object('analysis/monthly_source');state['prior_Recorder_object_loads']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');assert len(old_table)==60 and int(old_table.status.eq('current_original_source_semantics_and_clock_qualified').sum())==41
calendar=[str(x)[:10] for x in D.calendar(start_time='2018-01-01',end_time='2023-04-30',freq='day')];mon=['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER'];rows=[];errors=[]
for month in old_table.economic_month.tolist():
 r=new.get(month,by_month[month]);if_new=month in new
 if not if_new:state['saved_PDF_reads']+=1
 path=root/r['local_path'] if r.get('local_path') else None;c={'economic_month':month,'source_sha256':r['body_sha256'],'source_bytes':r['body_bytes'],'source_kind':'transport_repair' if if_new else 'reused_first_pass','HTTP_status':r['http_status'],'release_date':None,'release_time_ET':None,'first_Qlib_session_strictly_after_release':None,'manufacturers_adjusted_current_preliminary_ratio':None,'manufacturers_adjusted_previous_revised_ratio':None,'current_byte_first_public_qualified':False,'server_Last_Modified_UTC':None,'server_Last_Modified_after_release_date':None,'status':'unknown'}
 transport_ok=bool(path and path.is_file() and r['http_status']==200 and (not if_new or r['curl_exit']==0) and path.stat().st_size<=2000000 and path.read_bytes().startswith(b'%PDF'))
 if transport_ok:
  try:
   t='\n'.join(pg.extract_text() or '' for pg in PdfReader(io.BytesIO(path.read_bytes())).pages);head=t[:1800].upper();date=re.search(r'(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+(\d{1,2}),\s+(20\d{2})',head);time_ok=bool(re.search(r'10:00\s*(?:A\.?M\.?)?\s*E(?:D|S)T',head));title_ok=mon[int(month[5:])-1]+' '+month[:4] in head
   if date and time_ok and title_ok:
    release=f'{int(date.group(3)):04d}-{mon.index(date.group(1))+1:02d}-{int(date.group(2)):02d}';c['release_date']=release;c['release_time_ET']='10:00';c['first_Qlib_session_strictly_after_release']=next((d for d in calendar if d>release),None)
   start=re.search(r'(?im)^\s*Adjusted\s*\d*\s*$',t);tail=t[start.end():] if start else '';adjusted=re.split(r'(?i)Not\s+Adjusted',tail,maxsplit=1)[0];matches=list(re.finditer(r'Manufacturers(?:²|3|2)?[^\d]{0,100}?((?:\d[\d,]*\s+){6})(\d+\.\d\d)\s+(\d+\.\d\d)\s+(\d+\.\d\d)',adjusted,re.I|re.S))
   if len(matches)==1:
    z=matches[0];c['manufacturers_adjusted_current_preliminary_ratio']=float(z.group(2));c['manufacturers_adjusted_previous_revised_ratio']=float(z.group(3));c['same_release_prior_year_ratio']=float(z.group(4))
   else:c['parse_diagnostic']={'adjusted_Manufacturers_matches':len(matches),'adjusted_chars':len(adjusted),'release_date_found':bool(date),'clock_found':time_ok,'title_found':title_ok}
   if c['release_date'] and c['first_Qlib_session_strictly_after_release'] and c['manufacturers_adjusted_current_preliminary_ratio'] is not None:c['status']='current_original_source_semantics_and_clock_qualified'
  except Exception as e:c['parse_error']=type(e).__name__+': '+str(e)[:180]
 else:c['parse_error']='transport_or_PDF_signature_incomplete'
 if r.get('headers_path'):
  raw=(root/r['headers_path']).read_text(errors='replace');lm=re.findall(r'(?im)^Last-Modified:\s*(.+?)\s*$',raw)
  if lm:
   c['server_Last_Modified_UTC']=lm[-1]
   if c['release_date']:
    try:c['server_Last_Modified_after_release_date']=email.utils.parsedate_to_datetime(lm[-1]).date().isoformat()>c['release_date']
    except Exception:pass
 old=old_table.loc[old_table.economic_month.eq(month)].iloc[0]
 if old.status=='current_original_source_semantics_and_clock_qualified':
  for key in ['source_sha256','release_date','first_Qlib_session_strictly_after_release','manufacturers_adjusted_current_preliminary_ratio','manufacturers_adjusted_previous_revised_ratio','status']:assert c[key]==getattr(old,key),(month,key,c[key],getattr(old,key))
 rows.append(c)
 if c['status']=='unknown':errors.append({'month':month,'reason':c.get('parse_error'),'diagnostic':c.get('parse_diagnostic'),'http_status':r['http_status'],'transport_error':r.get('error')})
table=pd.DataFrame(rows);assert len(table)==60 and state['external_GETs_started']==3 and state['saved_PDF_reads']==57;complete=not errors;result={'status':'current_original_MTiS_2018_2022_60_month_source_complete_PIT_pending' if complete else 'current_original_MTiS_2018_2022_repair_still_unknown','plan_sha256':sha(card/'plan.json'),'months':60,'external_GETs_started':3,'saved_PDF_reads':57,'prior_accepted_rows_unchanged':41,'qualified_months':int(table.status.eq('current_original_source_semantics_and_clock_qualified').sum()),'unknown_months':[x['month'] for x in errors],'unknown_reasons':errors,'current_bytes_first_public_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=sorted(x for x in snap.iterdir() if x.is_file());entries={str(x.relative_to(root)):sha(x) for x in files}
with tempfile.TemporaryDirectory(prefix='mtis-train-repair-') as tmp:
 archive=pathlib.Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(x,arcname=str(x.relative_to(root)),recursive=False)
 digest=sha(archive);nas=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert nas['sha256']==digest and nas['size_bytes']==archive.stat().st_size;remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='mtis-train-repair-restore-') as back:
  copy=pathlib.Path(back)/'source.tar.gz'
  with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True,timeout=90)
  assert sha(copy)==digest
  with tarfile.open(copy,'r:gz') as t:
   members={m.name:m for m in t.getmembers() if m.isfile()};assert set(members)==set(entries)
   for rel,h in entries.items():assert hashlib.sha256(t.extractfile(members[rel]).read()).hexdigest()==h
source_archive={**nas,'nas_path':remote,'files':len(files),'all_member_cold_restore_verified':True,'prior_source_archive_sha256':s['prior_source_NAS_sha256']};(card/'source_archive.json').write_text(json.dumps(source_archive,indent=2)+'\n')
with R.uri_context(str(card/'qlib_recorder')):
 with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
  rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive=source_archive);rec.save_objects(artifact_path='analysis',monthly_source=table,unknown_reasons=errors,result=result);rec.log_metrics(months=60,qualified_months=result['qualified_months'],new_GETs=3,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'source_archive.json','source_NAS_sha256':digest,'source_NAS_cold_restore_verified':True,'prior_source_NAS_sha256':s['prior_source_NAS_sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'qualified_months':result['qualified_months'],'unknown_months':result['unknown_months'],'external_GETs':3,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_source','analysis/unknown_reasons','analysis/result']},'limitations':['Current original-PDF bytes and Last-Modified do not prove historical first-public vintages.','Manufacturers is an aggregate nominal inventory/sales measure, not semiconductor-specific.','No held source, Stage1 information screen, 0bp account or SOXX success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':result['status'],'qualified':result['qualified_months'],'unknown':result['unknown_months'],'recorder_id':rid},ensure_ascii=False))


2026/10/23 15:07:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[15095:MainThread](2026-10-23 15:07:45,042) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[15095:MainThread](2026-10-23 15:07:45,046) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[15095:MainThread](2026-10-23 15:07:45,047) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[15095:MainThread](2026-10-23 15:07:51,258) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_2018_2022_source_repair_20261023.


[15095:MainThread](2026-10-23 15:07:51,261) INFO - qlib.workflow - [exp.py:258] - Experiment 924953321268355477 starts running ...


[15095:MainThread](2026-10-23 15:07:51,343) INFO - qlib.workflow - [recorder.py:345] - Recorder 15a0168dc73c483bb93b68ba4f70844a starts running under Experiment 924953321268355477 ...


[15095:MainThread](2026-10-23 15:07:52,163) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "current_original_MTiS_2018_2022_60_month_source_complete_PIT_pending", "qualified": 60, "unknown": [], "recorder_id": "15a0168dc73c483bb93b68ba4f70844a"}


# 结论：Census MTIS 2018–2022 制造业合计库存销售比原始来源

完整 Stage1 f696+b472 仍是研究父方案，SOXX 是首个费用后0bp经济硬门。原版本59次GET加1份旧原件只解析41/60月；本修复只处理已查明的16个月“Not Adjusted”大小写差异，并对3份超时PDF各按相同官方URL重取一次。旧版41条已接受记录逐字段完全不变。

结果是 **60/60 当前官方原稿**各且仅有一条 Table1 **Adjusted Manufacturers** 当期初值、同版上期修订库存销售比和印刷10:00 ET发布日期；均保守映射到该发布日期之后首个美国Qlib交易日。原件与头、修复收据和 Qlib Recorder `15a0168dc73c483bb93b68ba4f70844a` 分别在NAS冷恢复。59份本批原件有HTTP Last-Modified，没有一份日期晚于印刷发布日期；另一个2018-07样本的旧收据为当日14:00:03 GMT。**这些可变的服务器元数据仍不足以证明历史首次公开字节。**

决定 `current_original_MTiS_2018_2022_60_month_source_complete_PIT_pending`：来源数值与保守日历可用性具备开发筛查的条件，但不是已认证的历史PIT面板。全美制造业合计比率并非半导体专属数据，且为名义价格口径。2023–2026留出期原件、修订版本核验、Stage1增量信息、完整0bp自身状态账户、2/5bp与独立前向均尚未完成。本卡0 SOXX结果读取、0拟合、0账户。
